# Data audit


## Inputs and scope

Inventory the CGMacros v1.0.0 release, reconcile dynamic and static subject IDs, and check schemas, coverage and release checksums. `source_manifest.csv` records dataset-relative paths and sizes; root-file hashes are compared with the released `SHA256SUMS.txt`. Participant CSVs and photos are not individually covered by that checksum manifest.


In [1]:
# --- Import dependencies and configure the audit output directory ---

import re
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from src.paths import CGMACROS_DIR, DATA_ROOT, artifact_dir, validate_data_root

validate_data_root()
DATA_QUALITY_DIR = artifact_dir("data_quality")

In [2]:
# --- Helpers for the input inventory and source manifest ---

def sha256_of_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Compute the SHA-256 hash of a file without loading it fully into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)

    return digest.hexdigest()

def human_size(num_bytes: int) -> str:
    """Convert a file size in bytes to a human-readable representation."""
    size = float(num_bytes)

    for unit in ["B", "KB", "MB", "GB"]:
        if size < 1024:
            return f"{size:.1f} {unit}"
        size /= 1024

    return f"{size:.1f} TB"

def relative_to_data_root(path: Path) -> str:
    """Return a path relative to the dataset root."""
    return path.relative_to(DATA_ROOT).as_posix()

# File metadata are accumulated here before constructing `source_manifest`.
manifest_rows = []
def add_manifest_row(
    name,
    path,
    category,
    role,
    status,
    size_bytes=None,
    sha256=None,
    notes="",
):
    """Append one source-file record to the manifest."""
    manifest_rows.append(
        {
            "name": name,
            "relative_path": relative_to_data_root(path) if path is not None else "",
            "category": category,
            "role": role,
            "status": status,
            "size": human_size(size_bytes) if size_bytes is not None else "",
            "sha256": sha256 or "",
            "notes": notes,
        }
    )

In [3]:
# --- Dataset-level files distributed alongside CGMacros/: data dictionaries, checksum manifest, license ---

# Names are enumerated explicitly (not just "list the folder") so a silently missing
# expected file is reported as MISSING rather than simply absent from the output.

root_files_to_catalog = [
    ("DataDictionary.pdf", "data_dictionary", "field dictionary (PDF)"),
    ("DataDictionary_Bio.csv", "data_dictionary", "Bio field dictionary"),
    ("DataDictionary_CGMacros-00X.csv", "data_dictionary", "dynamic-file field dictionary"),
    ("DataDictionary_Gut_Health_Test.csv", "data_dictionary", "Gut Health Test field dictionary"),
    ("DataDictionary_Microbes.csv", "data_dictionary", "Microbiome field dictionary"),
    ("SHA256SUMS.txt", "checksum_manifest", "release-package checksum listing"),
    ("LICENSE.txt", "license", "dataset license text"),
]

root_file_sha256 = {}
for filename, category, role in root_files_to_catalog:
    p = DATA_ROOT / filename
    if p.exists():
        root_file_sha256[filename] = sha256_of_file(p)
        add_manifest_row(
            name=filename, path=p, category=category, role=role, status="present_on_disk",
            size_bytes=p.stat().st_size, sha256=root_file_sha256[filename],
        )
    else:
        add_manifest_row(
            name=filename, path=p, category=category, role=role, status="MISSING",
            notes="Expected alongside the dataset but not found.",
        )

In [4]:
# --- Parse SHA256SUMS.txt and cross-check recorded hashes against files actually on disk ---

# Format observed: "<64-hex-char hash>  <filename>" (whitespace-separated).
sha_manifest_path = DATA_ROOT / "SHA256SUMS.txt"
recorded_hashes = {}
with open(sha_manifest_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(None, 1)
        if len(parts) == 2:
            recorded_hash, fname = parts
            recorded_hashes[fname.strip()] = recorded_hash.strip()

checksum_rows = []
for fname, recorded_hash in recorded_hashes.items():
    p = DATA_ROOT / fname
    present = p.exists()
    actual_hash = (root_file_sha256.get(fname) or sha256_of_file(p)) if present else None
    checksum_rows.append({
        "file": fname,
        "listed_in_SHA256SUMS": True,
        "present_on_disk": present,
        "recorded_hash": recorded_hash,
        "actual_hash": actual_hash,
        "hash_matches": (actual_hash.lower() == recorded_hash.lower()) if present else None,
    })

checksum_df = pd.DataFrame(checksum_rows)

# The release archive listed in SHA256SUMS.txt is not part of the extracted dataset, so absent files are
# reported in the table; a file that is present but differs from its recorded hash is an error.
mismatched_files = [row["file"] for row in checksum_rows if row["present_on_disk"] and not row["hash_matches"]]
if mismatched_files:
    raise ValueError(
        f"Raw files differ from SHA256SUMS.txt: {mismatched_files}. "
        "Restore the original CGMacros v1.0.0 files (see data/README.md)."
    )
checksum_df

,file,listed_in_SHA256SUMS,present_on_disk,recorded_hash,actual_hash,hash_matches
0,CGMacros_dateshifted365.zip,True,False,05c8b0e6f1a2757050aced55ce4bf6ab2ac9b30f2fd8ca...,None,None
1,DataDictionary.pdf,True,True,f161ff8df4bd95efa79d3daa8aba5eab754df287ab0f4d...,f161ff8df4bd95efa79d3daa8aba5eab754df287ab0f4d...,True
2,DataDictionary_Bio.csv,True,True,2d5b4c27873284ddbf218df51172b69ffa9caf6e97a3a1...,2d5b4c27873284ddbf218df51172b69ffa9caf6e97a3a1...,True
3,DataDictionary_CGMacros-00X.csv,True,True,db7bf114d61b90fc5de43fed5e5cd6ba33af37e900a706...,db7bf114d61b90fc5de43fed5e5cd6ba33af37e900a706...,True
4,DataDictionary_Gut_Health_Test.csv,True,True,fbc7555aacb93b2fb107b65db8d4a3d88d3546e7818fe5...,fbc7555aacb93b2fb107b65db8d4a3d88d3546e7818fe5...,True
5,DataDictionary_Microbes.csv,True,True,eb75be859489f59f6b81c3f8d29c307f078498655b4ae0...,eb75be859489f59f6b81c3f8d29c307f078498655b4ae0...,True
6,LICENSE.txt,True,True,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,True


## Dynamic per-participant data: directory, file, and header inventory

Each participant directory must contain its dynamic CSV and a `photos/` folder. Every CSV is loaded once here, as released, and the loaded frames are reused by the audits below.

In [5]:
# --- Enumerate participant directories under CGMacros/ and extract IDs from the actual folder names ---

# IDs are parsed from the directory names present on disk, not assumed in advance.
participant_dir_pattern = re.compile(r"^CGMacros-(\d+)$")

participant_dirs = []
for entry in sorted(CGMACROS_DIR.iterdir()):
    if entry.is_dir():
        m = participant_dir_pattern.match(entry.name)
        if m:
            participant_dirs.append((int(m.group(1)), entry))
        else:
            print(f"NOTE: directory does not match CGMacros-NNN pattern: {entry.name}")

dynamic_dir_ids = sorted(pid for pid, _ in participant_dirs)
print(f"Participant directories matching CGMacros-NNN: {len(participant_dirs)} (IDs {min(dynamic_dir_ids)}-{max(dynamic_dir_ids)})")

# CGMacros v1.0.0 releases 45 participants: IDs 1-49 excluding 24, 25, 37 and 40.
expected_cohort_ids = set(range(1, 24)) | set(range(26, 37)) | set(range(38, 40)) | set(range(41, 50))
if set(dynamic_dir_ids) != expected_cohort_ids:
    raise ValueError(
        "Participant directories do not match the CGMacros v1.0.0 cohort: "
        f"missing {sorted(expected_cohort_ids - set(dynamic_dir_ids))}, "
        f"unexpected {sorted(set(dynamic_dir_ids) - expected_cohort_ids)}."
    )

Participant directories matching CGMacros-NNN: 45 (IDs 1-49)


In [6]:
# --- For each participant: verify the CSV + photos/ structure, and load the CSV once ---

# CSVs are loaded as released (no type conversion or missing-value handling); the frames in
# `participant_dfs` are reused by every later audit step in this notebook.
participant_dfs = {}
participant_inventory_rows = []

for pid, dir_path in participant_dirs:
    # Build the expected participant CSV and photo-directory paths.
    padded = f"{pid:03d}"
    expected_csv_name = f"CGMacros-{padded}.csv"
    csv_path = dir_path / expected_csv_name
    photos_dir = dir_path / "photos"

    # Check whether the expected files and directories exist.
    entries_in_dir = sorted(p.name for p in dir_path.iterdir())
    unexpected_entries = [e for e in entries_in_dir if e not in {expected_csv_name, "photos"}]

    csv_exists = csv_path.exists()
    photos_exists = photos_dir.exists() and photos_dir.is_dir()
    photo_count = len(list(photos_dir.iterdir())) if photos_exists else 0

    n_rows, n_cols, columns = None, None, None
    if csv_exists:
        participant_dfs[pid] = pd.read_csv(csv_path)
        columns = tuple(participant_dfs[pid].columns)
        n_cols = len(columns)
        n_rows = len(participant_dfs[pid])

    # Store this participant's inventory record.
    participant_inventory_rows.append({
        "subject_id": pid,
        "dir_name": dir_path.name,
        "csv_exists": csv_exists,
        "photos_dir_exists": photos_exists,
        "photo_count": photo_count,
        "n_rows": n_rows,
        "n_cols": n_cols,
        "columns": columns,
        "unexpected_entries_in_dir": unexpected_entries,
    })

participant_inventory = pd.DataFrame(participant_inventory_rows).set_index("subject_id").sort_index()

missing_csv_subjects = participant_inventory.index[~participant_inventory["csv_exists"]].tolist()
if missing_csv_subjects:
    raise FileNotFoundError(f"Dynamic CSV missing for participant(s): {missing_csv_subjects}")

# Report structural coverage across the cohort.
print(f"Participants inventoried: {len(participant_inventory)}; total rows: {int(participant_inventory['n_rows'].sum())}; "
      f"photos/ present for all: {participant_inventory['photos_dir_exists'].all()}; "
      f"unexpected extra entries: {any(len(x) > 0 for x in participant_inventory['unexpected_entries_in_dir'])}")

# Summarize participants sharing each column-header signature.
display(participant_inventory.groupby("columns").size().rename("participants").to_frame())

participant_inventory[["csv_exists", "photos_dir_exists", "photo_count", "n_rows", "n_cols"]]

Participants inventoried: 45; total rows: 687580; photos/ present for all: True; unexpected extra entries: False


,csv_exists,photos_dir_exists,photo_count,n_rows,n_cols
subject_id,,,,,
1,True,True,90,14730,15
2,True,True,87,17025,15
3,True,True,68,14565,15
4,True,True,125,14275,15
5,True,True,96,14460,15
6,True,True,77,14460,15
7,True,True,64,5655,15
8,True,True,64,14760,14
9,True,True,82,14370,14


In [7]:
# --- Add one manifest row per participant's dynamic CSV and one per photos/ folder ---

# Individual photo files are intentionally NOT hashed or listed one-by-one; the photos/
# folder is recorded as a single inventory unit with a count.
for pid, dir_path in participant_dirs:
    row = participant_inventory.loc[pid]
    csv_path = dir_path / f"CGMacros-{pid:03d}.csv"
    photos_dir = dir_path / "photos"

    add_manifest_row(
        name=f"CGMacros-{pid:03d}.csv", path=csv_path, category="dynamic_participant_data",
        role="per-participant dynamic CGM/meal/activity time series", status="present_on_disk",
        size_bytes=csv_path.stat().st_size,
        notes=f"subject_id={pid}; rows={row['n_rows']}; cols={row['n_cols']}.",
    )

    add_manifest_row(
        name=f"CGMacros-{pid:03d}/photos", path=photos_dir, category="photo_evidence",
        role="meal/event photo references for this participant",
        status="present_on_disk" if row["photos_dir_exists"] else "MISSING",
        notes=f"subject_id={pid}; photo_count={row['photo_count']}. Individual photos are not hashed or listed.",
    )

## Static tables and data dictionaries

The three static tables (`bio.csv`, `gut_health_test.csv`, `microbes.csv`) are read to confirm shape and extract `subject` identifiers; the four data dictionaries are read once and reused by the schema checks below.

In [8]:
# --- Static tables: bio.csv, gut_health_test.csv, microbes.csv ---

static_files = {
    "bio.csv": "Bio",
    "gut_health_test.csv": "Gut Health Test",
    "microbes.csv": "Microbiome",
}

static_tables = {}
static_summary_rows = []

for filename, label in static_files.items():
    p = CGMACROS_DIR / filename
    df = pd.read_csv(p, encoding="utf-8-sig")
    static_tables[filename] = df

    subject_col_present = "subject" in df.columns
    subject_ids = sorted(df["subject"].dropna().unique().tolist()) if subject_col_present else None
    n_unique_subjects = len(subject_ids) if subject_ids is not None else None

    add_manifest_row(
        name=filename, path=p, category="static_participant_data", role=f"{label} static table",
        status="present_on_disk", size_bytes=p.stat().st_size,
        notes=f"shape={df.shape}; subject column present={subject_col_present}; "
              f"unique subjects={n_unique_subjects}.",
    )

    static_summary_rows.append({
        "file": filename, "label": label, "n_rows": df.shape[0], "n_cols": df.shape[1],
        "subject_col_present": subject_col_present, "n_unique_subjects": n_unique_subjects,
    })

static_summary = pd.DataFrame(static_summary_rows)
static_summary

,file,label,n_rows,n_cols,subject_col_present,n_unique_subjects
0,bio.csv,Bio,45,24,True,45
1,gut_health_test.csv,Gut Health Test,47,23,True,47
2,microbes.csv,Microbiome,45,1980,True,45


In [9]:
# --- Data dictionary field-count check ---

dictionary_files = [
    ("DataDictionary_CGMacros-00X.csv", "dynamic fields"),
    ("DataDictionary_Bio.csv", "Bio fields"),
    ("DataDictionary_Gut_Health_Test.csv", "Gut Health Test fields"),
    ("DataDictionary_Microbes.csv", "Microbiome fields"),
]
data_dictionaries = {}
dict_rows = []

for filename, label in dictionary_files:
    path = DATA_ROOT / filename
    df = pd.read_csv(path, encoding="utf-8-sig")
    data_dictionaries[filename] = df

    dict_rows.append({
        "file": filename,
        "label": label,
        "field_count": df.shape[0],
        "columns": list(df.columns),
    })

dict_check_df = pd.DataFrame(dict_rows)
dict_check_df

,file,label,field_count,columns
0,DataDictionary_CGMacros-00X.csv,dynamic fields,14,"[Field, Range, Description]"
1,DataDictionary_Bio.csv,Bio fields,23,"[Field, Range, Description]"
2,DataDictionary_Gut_Health_Test.csv,Gut Health Test fields,22,"[Field, Value, Description (from Viome)]"
3,DataDictionary_Microbes.csv,Microbiome fields,1979,"[Field, Range, Description]"


## Cohort reconciliation: dynamic ids vs. static `subject` ids

Participant identifiers from the dynamic directories are cross-checked against the `subject` column of each static table.

In [10]:
# --- Build ID sets from the actual data ---

dynamic_ids = set(dynamic_dir_ids)  # from CGMacros-NNN/ directory names, computed above
bio_ids = set(static_tables["bio.csv"]["subject"].dropna().astype(int).unique())
gut_ids = set(static_tables["gut_health_test.csv"]["subject"].dropna().astype(int).unique())
microbes_ids = set(static_tables["microbes.csv"]["subject"].dropna().astype(int).unique())

# Compare ID-set sizes across the four sources.
cohort_summary = pd.DataFrame({
    "source": ["dynamic (directories)", "bio.csv", "gut_health_test.csv", "microbes.csv"],
    "n_ids": [len(dynamic_ids), len(bio_ids), len(gut_ids), len(microbes_ids)],
})
# Build a per-subject membership matrix across all four sources.
all_ids = dynamic_ids | bio_ids | gut_ids | microbes_ids
cohort_membership = pd.DataFrame({
    "subject_id": sorted(all_ids),
}).set_index("subject_id")
cohort_membership["in_dynamic"] = cohort_membership.index.isin(dynamic_ids)
cohort_membership["in_bio"] = cohort_membership.index.isin(bio_ids)
cohort_membership["in_gut"] = cohort_membership.index.isin(gut_ids)
cohort_membership["in_microbes"] = cohort_membership.index.isin(microbes_ids)

# IDs that appear in every source vs. IDs that appear in at least one static table but
# were never seen as a dynamic participant (or vice versa) are the key facts of interest.
in_all_four = cohort_membership.all(axis=1)
cohort_summary["missing_from_dynamic"] = [0, len(bio_ids - dynamic_ids), len(gut_ids - dynamic_ids), len(microbes_ids - dynamic_ids)]
cohort_summary["missing_from_source"] = [0, len(dynamic_ids - bio_ids), len(dynamic_ids - gut_ids), len(dynamic_ids - microbes_ids)]
print(cohort_summary.to_string(index=False))
print(f"Subjects present in all four sources: {int(in_all_four.sum())}/{len(all_ids)}")

               source  n_ids  missing_from_dynamic  missing_from_source
dynamic (directories)     45                     0                    0
              bio.csv     45                     0                    0
  gut_health_test.csv     47                     2                    0
         microbes.csv     45                     0                    0
Subjects present in all four sources: 45/47


In [11]:
# --- Assemble source_manifest from the release files cataloged above ---

source_manifest = pd.DataFrame(manifest_rows)

# Separately flag and list any rows marked MISSING, if there are any.
missing_rows = source_manifest[source_manifest["status"] == "MISSING"]
print(f"Rows flagged MISSING: {len(missing_rows)}")
if len(missing_rows) > 0:
    print(missing_rows[["name", "category", "notes"]].to_string(index=False))

# --- Persist source_manifest ---

source_manifest_path = DATA_QUALITY_DIR / "source_manifest.csv"
source_manifest.to_csv(source_manifest_path, index=False)

Rows flagged MISSING: 0


The header summary shows multiple raw schemas; `02_clean_align.ipynb` harmonizes them.


## Schema and data quality audit


Audit the 45 dynamic CSVs against `DataDictionary_CGMacros-00X.csv`. Nothing is imputed, standardized or removed; `METs` and `Intensity` stay distinct raw columns, and index-artifact columns (`Unnamed: 0`, `RecordIndex`) are never treated as physiological features.

Outputs: `schema_map.csv`, `quality_issues.csv`, `coverage_summary.csv`.

In [12]:
# --- Canonical field reference derived from DataDictionary_CGMacros-00X.csv ---

def normalize_column_name(name: str) -> str:
    """Lowercase and collapse whitespace, for name-matching purposes only (not for renaming data)."""
    return re.sub(r"\s+", " ", str(name).strip()).lower()


def extract_numeric_bounds(range_text: str):
    """Extract (low, high) from dictionary strings like 'Numeric: 40-400 mg/dL'; else (None, None)."""
    match = re.search(r"(-?\d+(?:\.\d+)?)\s*-\s*(-?\d+(?:\.\d+)?)", str(range_text))
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None


def find_column(df, normalized_name):
    """Return the first column of `df` whose normalized name equals `normalized_name`, or None."""
    return next((c for c in df.columns if normalize_column_name(c) == normalized_name), None)


# Load the field dictionary and derive normalized names and parsed numeric ranges.
dynamic_dictionary_df = data_dictionaries["DataDictionary_CGMacros-00X.csv"].copy()
dynamic_dictionary_df["normalized_field"] = dynamic_dictionary_df["Field"].map(normalize_column_name)
dynamic_dictionary_df[["range_low", "range_high"]] = dynamic_dictionary_df["Range"].apply(
    lambda r: pd.Series(extract_numeric_bounds(r))
)

# normalized name -> the dictionary's own (canonical) spelling of the field
canonical_lookup = dict(zip(dynamic_dictionary_df["normalized_field"], dynamic_dictionary_df["Field"]))
# canonical (dictionary) field name -> (low, high) numeric bound, where parseable
canonical_ranges = dict(zip(
    dynamic_dictionary_df["Field"],
    zip(dynamic_dictionary_df["range_low"], dynamic_dictionary_df["range_high"]),
))


In [13]:
# --- Build schema_map: one row per distinct raw column name observed across the cohort ---

# Columns that are pandas export/record-index artifacts, never physiological measurements.
INDEX_ARTIFACT_NAMES = {"unnamed: 0", "recordindex"}
# Present in some files but absent from DataDictionary_CGMacros-00X.csv; kept as separate,
# undocumented raw columns rather than assumed equivalents of a documented field.
KNOWN_UNDOCUMENTED_ACTIVITY_NAMES = {"intensity", "steps"}
KNOWN_UNDOCUMENTED_NUTRITION_NAMES = {"sugar"}


def classify_column(normalized_name, dictionary_match):
    if dictionary_match is not None:
        return "dictionary_defined"
    if normalized_name in INDEX_ARTIFACT_NAMES:
        return "index_artifact_not_physiological"
    if normalized_name in KNOWN_UNDOCUMENTED_ACTIVITY_NAMES:
        return "undocumented_activity_field"
    if normalized_name in KNOWN_UNDOCUMENTED_NUTRITION_NAMES:
        return "undocumented_nutrition_field"
    return "undocumented_other"


# Tally how many participants show each raw column name, and match it to the dictionary.
schema_records = {}
for pid, df in participant_dfs.items():
    for col in df.columns:
        norm = normalize_column_name(col)
        record = schema_records.setdefault(col, {
            "raw_column_name": col,
            "normalized_name": norm,
            "n_participants_with_exact_name": 0,
            "dictionary_field_match": canonical_lookup.get(norm),
        })
        record["n_participants_with_exact_name"] += 1

# Turn the tally into one row per distinct column, with its match type and category.
schema_map_rows = []
for record in schema_records.values():
    dictionary_match = record["dictionary_field_match"]
    if dictionary_match is not None:
        match_type = "exact" if record["raw_column_name"] == dictionary_match else "case_or_space_difference"
    else:
        match_type = None
    schema_map_rows.append({
        **record,
        "match_type": match_type,
        "category": classify_column(record["normalized_name"], dictionary_match),
    })
schema_map = pd.DataFrame(schema_map_rows).sort_values("raw_column_name").reset_index(drop=True)

print(f"Distinct raw column names observed across the cohort: {len(schema_map)}")
print(schema_map["category"].value_counts().to_string())
schema_map

Distinct raw column names observed across the cohort: 20
category
dictionary_defined                  15
undocumented_activity_field          2
index_artifact_not_physiological     2
undocumented_nutrition_field         1


,raw_column_name,normalized_name,n_participants_with_exact_name,dictionary_field_match,match_type,category
0,Amount Consumed,amount consumed,42,Amount Consumed,exact,dictionary_defined
1,Amount Consumed,amount consumed,1,Amount Consumed,case_or_space_difference,dictionary_defined
2,Calories,calories,45,Calories,exact,dictionary_defined
3,Calories (Activity),calories (activity),44,Calories (Activity),exact,dictionary_defined
4,Carbs,carbs,45,Carbs,exact,dictionary_defined
5,Dexcom GL,dexcom gl,45,Dexcom GL,exact,dictionary_defined
6,Fat,fat,45,Fat,exact,dictionary_defined
7,Fiber,fiber,45,Fiber,exact,dictionary_defined
8,HR,hr,45,HR,exact,dictionary_defined
9,Image path,image path,45,Image Path,case_or_space_difference,dictionary_defined


In [14]:
# --- Confirm METs and Intensity never co-occur in the same participant file ---

mets_participants = {pid for pid, df in participant_dfs.items()
                     if any(normalize_column_name(c) == "mets" for c in df.columns)}
intensity_participants = {pid for pid, df in participant_dfs.items()
                          if any(normalize_column_name(c) == "intensity" for c in df.columns)}
both_present = mets_participants & intensity_participants

print(f"Participants with a 'METs' column     : {len(mets_participants)}")
print(f"Participants with an 'Intensity' column: {len(intensity_participants)}")
print(f"Participants with BOTH in the same file: {len(both_present)}")

Participants with a 'METs' column     : 34
Participants with an 'Intensity' column: 11
Participants with BOTH in the same file: 0


In [15]:
# --- Missing dictionary columns, and missingness for columns that ARE present ---

canonical_fields_normalized = set(canonical_lookup.keys())
missing_column_rows = []
column_missingness_rows = []

for pid, df in participant_dfs.items():
    n_rows = len(df)
    present_normalized = {normalize_column_name(c) for c in df.columns}

    # Flag dictionary-defined columns entirely absent from this participant's file.
    for norm_field in sorted(canonical_fields_normalized - present_normalized):
        missing_column_rows.append({
            "subject_id": pid,
            "issue_type": "missing_column",
            "column": canonical_lookup[norm_field],
            "detail": "dictionary-defined column absent from this participant's file",
        })

    # Count missing values in each column that is present, including fully-empty columns.
    for col in df.columns:
        n_missing = int(df[col].isna().sum())
        if n_missing == 0:
            continue
        fully_missing = n_rows > 0 and n_missing == n_rows
        column_missingness_rows.append({
            "subject_id": pid,
            "issue_type": "fully_missing_column" if fully_missing else "column_has_missing_values",
            "column": col,
            "detail": f"{n_missing}/{n_rows} rows missing ({100 * n_missing / n_rows:.1f}%)",
            "n_missing": n_missing,
            "n_rows": n_rows,
        })

print(f"Missing columns: {len(missing_column_rows)}; columns with missing values: {len(column_missingness_rows)}; "
      f"fully missing: {sum(r['issue_type'] == 'fully_missing_column' for r in column_missingness_rows)}")

Missing columns: 14; columns with missing values: 499; fully missing: 0


In [16]:
# --- Event-field blankness: these fields are populated only at meal-log rows, so a
#     high blank rate is expected by design, not necessarily a data-quality defect ---

EVENT_TRIGGERED_FIELDS_NORMALIZED = {
    normalize_column_name(name) for name in
    ["Meal Type", "Calories", "Carbs", "Protein", "Fat", "Fiber", "Amount Consumed", "Image Path"]
}

event_field_blankness_rows = []
for pid, df in participant_dfs.items():
    n_rows = len(df)
    for col in df.columns:
        if normalize_column_name(col) not in EVENT_TRIGGERED_FIELDS_NORMALIZED:
            continue
        n_blank = int(df[col].isna().sum())
        event_field_blankness_rows.append({
            "subject_id": pid,
            "column": col,
            "n_blank": n_blank,
            "n_rows": n_rows,
            "pct_blank": round(100 * n_blank / n_rows, 2) if n_rows else None,
        })

event_field_blankness = pd.DataFrame(event_field_blankness_rows)

print("Median blank rate by field, across participants (%):")
print(event_field_blankness.groupby("column")["pct_blank"].median().round(1).to_string())

Median blank rate by field, across participants (%):
column
Amount Consumed     99.8
Amount Consumed     99.7
Calories            99.8
Carbs               99.8
Fat                 99.8
Fiber               99.8
Image path          99.5
Meal Type           99.8
Protein             99.8


In [17]:
# --- Timestamp duplicates / reversed order / sampling interval / long gaps ---

LONG_GAP_THRESHOLDS_MINUTES = [15, 60, 1440]  # audit-only thresholds, not a cleaning cutoff

parsed_timestamps = {}
timestamp_quality_rows = []
participant_timestamp_summary_rows = []

for pid, df in participant_dfs.items():
    # Locate the Timestamp column; record and skip participants missing it entirely.
    ts_col = find_column(df, "timestamp")
    if ts_col is None:
        timestamp_quality_rows.append({
            "subject_id": pid, "issue_type": "timestamp_column_missing", "column": None,
            "detail": "no column matched 'Timestamp'",
        })
        continue

    # Parse timestamps and flag any values that failed to parse.
    parsed = pd.to_datetime(df[ts_col], errors="coerce")
    parsed_timestamps[pid] = parsed
    n_unparsed = int(parsed.isna().sum() - df[ts_col].isna().sum())
    if n_unparsed > 0:
        timestamp_quality_rows.append({
            "subject_id": pid, "issue_type": "timestamp_unparseable", "column": ts_col,
            "detail": f"{n_unparsed} non-null raw values failed to parse as a timestamp",
        })

    # Check for duplicate timestamp values.
    valid = parsed.dropna()
    n_duplicates = int(valid.duplicated().sum())
    if n_duplicates:
        timestamp_quality_rows.append({
            "subject_id": pid, "issue_type": "timestamp_duplicate", "column": ts_col,
            "detail": f"{n_duplicates} duplicate timestamp values (original row order)",
        })

    # Check whether row order ever goes backwards in time.
    diffs = valid.diff().dropna()
    n_reversed = int((diffs < pd.Timedelta(0)).sum())
    if n_reversed:
        timestamp_quality_rows.append({
            "subject_id": pid, "issue_type": "timestamp_reversed", "column": ts_col,
            "detail": f"{n_reversed} rows where the timestamp decreases vs. the previous row (original order)",
        })

    # Compute the modal sampling interval and the largest observed gap.
    positive_diffs_minutes = diffs[diffs > pd.Timedelta(0)].dt.total_seconds() / 60
    modal_interval_minutes = (
        positive_diffs_minutes.round(2).mode().iloc[0] if not positive_diffs_minutes.empty else None
    )
    max_gap_minutes = positive_diffs_minutes.max() if not positive_diffs_minutes.empty else None

    # Count gaps that exceed each audit threshold.
    long_gap_counts = {}
    for threshold in LONG_GAP_THRESHOLDS_MINUTES:
        n_over = int((positive_diffs_minutes > threshold).sum())
        long_gap_counts[threshold] = n_over
        if n_over:
            timestamp_quality_rows.append({
                "subject_id": pid, "issue_type": "long_gap", "column": ts_col,
                "detail": f"{n_over} gaps exceeding {threshold} minutes (audit threshold, not a cleaning cutoff)",
            })

    # Store this participant's timestamp-quality summary.
    participant_timestamp_summary_rows.append({
        "subject_id": pid,
        "n_valid_timestamps": len(valid),
        "n_duplicate_timestamps": n_duplicates,
        "n_reversed_pairs": n_reversed,
        "modal_interval_minutes": modal_interval_minutes,
        "max_gap_minutes": round(max_gap_minutes, 1) if max_gap_minutes is not None else None,
        **{f"n_gaps_over_{t}min": v for t, v in long_gap_counts.items()},
    })

participant_timestamp_summary = pd.DataFrame(participant_timestamp_summary_rows).set_index("subject_id")

print("Modal sampling interval (minutes) across the cohort:")
print(participant_timestamp_summary["modal_interval_minutes"].value_counts().to_string())
print(f"Timestamp issues: duplicate subjects={(participant_timestamp_summary['n_duplicate_timestamps'] > 0).sum()}, "
      f"reversed-order subjects={(participant_timestamp_summary['n_reversed_pairs'] > 0).sum()}, "
      f"maximum gap={participant_timestamp_summary['max_gap_minutes'].max()} min")

Modal sampling interval (minutes) across the cohort:
modal_interval_minutes
1.0    45
Timestamp issues: duplicate subjects=0, reversed-order subjects=0, maximum gap=901.0 min


In [18]:
# --- Anomaly candidates: values outside the dictionary-declared numeric range ---

anomaly_rows = []
for pid, df in participant_dfs.items():
    for col in df.columns:
        canonical_field = canonical_lookup.get(normalize_column_name(col))
        if canonical_field is None:
            continue
        low, high = canonical_ranges.get(canonical_field, (None, None))
        if low is None or high is None:
            continue
        candidates = pd.to_numeric(df[col], errors="coerce").dropna()
        n_below = int((candidates < low).sum())
        n_above = int((candidates > high).sum())
        if n_below or n_above:
            anomaly_rows.append({
                "subject_id": pid, "issue_type": "anomaly_candidate", "column": col,
                "detail": f"{n_below} below {low}, {n_above} above {high} "
                          f"(dictionary range for '{canonical_field}')",
                "n_below": n_below, "n_above": n_above,
            })

anomaly_by_column = (
    pd.DataFrame(anomaly_rows).groupby("column")[["n_below", "n_above"]].sum()
    if anomaly_rows else pd.DataFrame()
)
print(f"Anomaly-candidate records (participant x field with >=1 out-of-range value): {len(anomaly_rows)}")
print(anomaly_by_column.to_string())

Anomaly-candidate records (participant x field with >=1 out-of-range value): 105
                     n_below  n_above
column                               
Amount Consumed            0      131
Calories                  58       27
Calories (Activity)        0        2
Carbs                      0       11
Fat                        0        1
Fiber                      0       23
Libre GL                   0        9
METs                       3        0
Protein                  243        0


In [19]:
# --- Assemble quality_issues (tidy long format) and coverage_summary (per participant) ---

quality_issues = pd.DataFrame(
    missing_column_rows + column_missingness_rows + timestamp_quality_rows + anomaly_rows
)

# Tally missing-column and anomaly-candidate counts per subject, for joining below.
n_missing_columns_by_subject = (
    pd.Series([r["subject_id"] for r in missing_column_rows], dtype="int64")
    .value_counts()
)

# Build coverage_summary: per-participant row/column counts plus quality-issue tallies.
coverage_summary = participant_inventory[["n_rows", "n_cols"]].join(participant_timestamp_summary)
coverage_summary["n_missing_dictionary_columns"] = (
    n_missing_columns_by_subject.reindex(coverage_summary.index, fill_value=0)
)
coverage_summary["n_anomaly_candidate_fields"] = (
    pd.Series([r["subject_id"] for r in anomaly_rows], dtype="int64").value_counts()
    .reindex(coverage_summary.index, fill_value=0)
)

print(quality_issues["issue_type"].value_counts().to_string())


issue_type
column_has_missing_values    499
anomaly_candidate            105
missing_column                14
long_gap                       3


In [20]:
# Persist schema_map
schema_map_path = DATA_QUALITY_DIR / "schema_map.csv"
schema_map.to_csv(schema_map_path, index=False)

# Persist quality_issues
quality_issues_path = DATA_QUALITY_DIR / "quality_issues.csv"
quality_issues.to_csv(quality_issues_path, index=False)

# Persist coverage_summary, keeping its subject_id index
coverage_summary_path = DATA_QUALITY_DIR / "coverage_summary.csv"
coverage_summary.to_csv(coverage_summary_path, index=True)

## Device, meal, and photo coverage

Review device, meal and photo coverage. Meal construction and broken-reference repair occur during data preparation; the tables below identify affected records.


In [21]:
# --- Device coverage: row exists vs. device value observed vs. both-device overlap ---

# The release table is already a per-minute grid (modal interval = 1.0 min for all
# participants, see the timestamp audit above), so "row exists" is equivalent to
# "part of the released 1-minute series"; the meaningful distinction here is whether each
# device's own reading is non-null.

device_coverage_rows = []
for pid, df in participant_dfs.items():
    n_rows = len(df)
    libre_col = find_column(df, "libre gl")
    dexcom_col = find_column(df, "dexcom gl")
    libre_observed = df[libre_col].notna() if libre_col is not None else pd.Series(False, index=df.index)
    dexcom_observed = df[dexcom_col].notna() if dexcom_col is not None else pd.Series(False, index=df.index)
    both_observed = libre_observed & dexcom_observed

    device_coverage_rows.append({
        "subject_id": pid,
        "n_rows": n_rows,
        "n_libre_observed": int(libre_observed.sum()),
        "n_dexcom_observed": int(dexcom_observed.sum()),
        "n_both_observed": int(both_observed.sum()),
        "pct_libre_observed": round(100 * libre_observed.sum() / n_rows, 2) if n_rows else None,
        "pct_dexcom_observed": round(100 * dexcom_observed.sum() / n_rows, 2) if n_rows else None,
        "pct_both_observed": round(100 * both_observed.sum() / n_rows, 2) if n_rows else None,
    })

device_coverage = pd.DataFrame(device_coverage_rows).set_index("subject_id")

total_rows = int(device_coverage["n_rows"].sum())
total_libre = int(device_coverage["n_libre_observed"].sum())
total_dexcom = int(device_coverage["n_dexcom_observed"].sum())
total_both = int(device_coverage["n_both_observed"].sum())

print(f"Cohort totals -- rows: {total_rows}, Libre observed: {total_libre}, "
      f"Dexcom observed: {total_dexcom}, both observed: {total_both}")

Cohort totals -- rows: 687580, Libre observed: 687360, Dexcom observed: 629825, both observed: 629605


In [22]:
# --- Raw Meal Type category frequencies, case-sensitive, across the whole cohort ---

meal_type_value_counts = {}
for pid, df in participant_dfs.items():
    meal_col = find_column(df, "meal type")
    if meal_col is None:
        continue
    for value, count in df[meal_col].dropna().value_counts().items():
        meal_type_value_counts[value] = meal_type_value_counts.get(value, 0) + count

meal_type_summary = pd.Series(meal_type_value_counts, name="n_rows").sort_values(ascending=False)
print("Raw Meal Type categories observed across the cohort (case-sensitive):")
print(meal_type_summary.to_string())

Raw Meal Type categories observed across the cohort (case-sensitive):
dinner       418
snack        300
lunch        272
breakfast    266
Breakfast    170
Lunch        163
Dinner        74
Snacks        38
Snack          4
snack 1        1


In [23]:
# --- Meal-tagged row counts and adjacent-minute ("consecutive") meal entries ---

# This only detects the phenomenon; whether/how to merge such entries into a single
# meal episode is a later-stage decision, not made here.
meal_event_rows = []
for pid, df in participant_dfs.items():
    meal_col = find_column(df, "meal type")
    if meal_col is None:
        continue
    is_meal = df[meal_col].notna()
    n_meal_events = int(is_meal.sum())
    n_consecutive = int((is_meal & is_meal.shift(1, fill_value=False)).sum())
    meal_event_rows.append({
        "subject_id": pid,
        "n_meal_events": n_meal_events,
        "n_consecutive_meal_rows": n_consecutive,
    })

meal_event_summary = pd.DataFrame(meal_event_rows).set_index("subject_id")
print(f"Total meal-tagged rows across cohort: {int(meal_event_summary['n_meal_events'].sum())}")

Total meal-tagged rows across cohort: 1706


In [24]:
# --- Amount Consumed magnitude, among rows where a value is actually present ---

amount_consumed_rows = []
for pid, df in participant_dfs.items():
    ac_col = find_column(df, "amount consumed")
    if ac_col is None:
        continue
    values = pd.to_numeric(df[ac_col], errors="coerce").dropna()
    if values.empty:
        continue
    amount_consumed_rows.append({
        "subject_id": pid,
        "n_values": len(values),
        "min": values.min(),
        "median": values.median(),
        "max": values.max(),
        "n_above_100": int((values > 100).sum()),
    })

amount_consumed_summary = pd.DataFrame(amount_consumed_rows).set_index("subject_id")
print(amount_consumed_summary[["n_values", "min", "median", "max", "n_above_100"]].describe().to_string())
total_above_100 = int(amount_consumed_summary["n_above_100"].sum())
print(f"Total values >100 across cohort: {total_above_100}")

        n_values         min      median         max  n_above_100
count  43.000000   43.000000   43.000000   43.000000    43.000000
mean   38.186047   46.093023   81.581395  240.651163     3.046512
std    10.005370   46.104340   38.981212  244.951490     4.770712
min    17.000000    0.000000    1.000000    4.000000     0.000000
25%    31.000000    0.000000  100.000000  100.000000     0.000000
50%    39.000000   50.000000  100.000000  100.000000     0.000000
75%    43.000000  100.000000  100.000000  450.000000     5.000000
max    64.000000  100.000000  100.000000  900.000000    19.000000
Total values >100 across cohort: 131


In [25]:
# --- Assemble meal_issues (tidy long format) ---

CANONICAL_MEAL_NAMES = {"Breakfast", "Lunch", "Dinner"}
meal_issue_rows = []

# Flag raw Meal Type values that are not an exact match to the canonical set.
for raw_value, count in meal_type_summary.items():
    if raw_value not in CANONICAL_MEAL_NAMES:
        meal_issue_rows.append({
            "subject_id": None,
            "issue_type": "meal_type_variant",
            "detail": f"raw value '{raw_value}' ({int(count)} rows) is not an exact "
                      f"case-sensitive match to {sorted(CANONICAL_MEAL_NAMES)}",
        })

# Flag participants with adjacent-minute meal-tagged row pairs.
for pid, row in meal_event_summary.iterrows():
    if row["n_consecutive_meal_rows"] > 0:
        meal_issue_rows.append({
            "subject_id": pid,
            "issue_type": "consecutive_meal_rows",
            "detail": f"{int(row['n_consecutive_meal_rows'])} adjacent-minute meal-tagged row pairs",
        })

# Flag participants with Amount Consumed values exceeding the declared range.
for pid, row in amount_consumed_summary.iterrows():
    if row["n_above_100"] > 0:
        meal_issue_rows.append({
            "subject_id": pid,
            "issue_type": "amount_consumed_above_100",
            "detail": f"{int(row['n_above_100'])} of {int(row['n_values'])} meal-tagged values exceed 100%",
        })

meal_issues = pd.DataFrame(meal_issue_rows)
print(meal_issues["issue_type"].value_counts().to_string())

issue_type
amount_consumed_above_100    16
meal_type_variant             7
consecutive_meal_rows         5


In [26]:
# --- Photo reference existence check: valid vs. broken Image path references ---

broken_image_rows = []
photo_reference_rows = []

for pid, dir_path in participant_dirs:
    df = participant_dfs[pid]
    img_col = find_column(df, "image path")
    if img_col is None:
        continue
    referenced = df[img_col].dropna()
    for row_index, rel_path in referenced.items():
        resolved = dir_path / rel_path
        exists = resolved.exists()
        photo_reference_rows.append({"subject_id": pid, "row_index": row_index, "exists": exists})
        if not exists:
            broken_image_rows.append({
                "subject_id": pid,
                "row_index": row_index,
                "referenced_path": rel_path,
            })

broken_image_references = pd.DataFrame(broken_image_rows)
photo_reference_summary = pd.DataFrame(photo_reference_rows)

print(f"Image references: {len(photo_reference_summary)}; broken: {len(broken_image_references)}")



Image references: 3197; broken: 3


In [27]:
# --- Informative only: photo files present on disk but never referenced by any row ---

unreferenced_photo_rows = []
for pid, dir_path in participant_dirs:
    df = participant_dfs[pid]
    img_col = find_column(df, "image path")
    referenced_names = set(df[img_col].dropna()) if img_col is not None else set()

    photos_dir = dir_path / "photos"
    actual_files = {f"photos/{p.name}" for p in photos_dir.iterdir()} if photos_dir.exists() else set()

    unreferenced_photo_rows.append({
        "subject_id": pid,
        "n_actual_photos": len(actual_files),
        "n_referenced_and_present": len(referenced_names & actual_files),
        "n_unreferenced": len(actual_files - referenced_names),
    })

unreferenced_photo_summary = pd.DataFrame(unreferenced_photo_rows).set_index("subject_id")
print(f"Photo files: {int(unreferenced_photo_summary['n_actual_photos'].sum())}; "
      f"unreferenced: {int(unreferenced_photo_summary['n_unreferenced'].sum())}")

Photo files: 3454; unreferenced: 260


## Static data and subject integrity

Check static subject-key uniqueness, cohort membership, empty rows and dictionary coverage. Field conflicts are recorded for interpretation; unit conversion and feature construction occur during data preparation.


In [29]:
# --- Load the three static-table data dictionaries ---

def load_static_dictionary(filename, field_col="Field"):
    df = data_dictionaries[filename].copy()
    df["normalized_field"] = df[field_col].map(normalize_column_name)
    return df

bio_dictionary_df = load_static_dictionary("DataDictionary_Bio.csv")
gut_dictionary_df = load_static_dictionary("DataDictionary_Gut_Health_Test.csv")
microbes_dictionary_df = load_static_dictionary("DataDictionary_Microbes.csv")

In [30]:
# --- Subject-key uniqueness and shape of each static table ---

for name, df in static_tables.items():
    n_dup = int(df["subject"].duplicated().sum())
    if n_dup:
        raise ValueError(f"{name} has {n_dup} duplicate subject key(s); subject must identify one row.")

In [31]:
# --- Out-of-cohort subjects and fully-empty rows, per static table ---

static_integrity_rows = []
for name, df in static_tables.items():
    non_subject_fields = [c for c in df.columns if c != "subject"]
    fully_empty_mask = df[non_subject_fields].isna().all(axis=1)
    fully_empty_subjects = sorted(df.loc[fully_empty_mask, "subject"].tolist())
    out_of_cohort = sorted(set(df["subject"]) - dynamic_ids)

    static_integrity_rows.append({
        "table": name,
        "n_rows": len(df),
        "n_unique_subjects": df["subject"].nunique(),
        "out_of_cohort_subjects": out_of_cohort,
        "fully_empty_subjects": fully_empty_subjects,
    })

static_integrity_summary = pd.DataFrame(static_integrity_rows)
print(static_integrity_summary.to_string(index=False))

              table  n_rows  n_unique_subjects out_of_cohort_subjects fully_empty_subjects
            bio.csv      45                 45                     []                   []
gut_health_test.csv      47                 47               [24, 25] [24, 25, 26, 28, 48]
       microbes.csv      45                 45                     []                 [48]


In [32]:
# --- Cohort completeness (present in dynamic cohort AND not a fully-empty row) ---

gut_empty_set = set(static_integrity_summary.loc[
    static_integrity_summary["table"] == "gut_health_test.csv", "fully_empty_subjects"
].iloc[0])
microbes_empty_set = set(static_integrity_summary.loc[
    static_integrity_summary["table"] == "microbes.csv", "fully_empty_subjects"
].iloc[0])

gut_complete_in_cohort = (set(static_tables["gut_health_test.csv"]["subject"]) & dynamic_ids) - gut_empty_set
microbes_complete_in_cohort = (set(static_tables["microbes.csv"]["subject"]) & dynamic_ids) - microbes_empty_set
both_complete = gut_complete_in_cohort & microbes_complete_in_cohort

print(f"Gut AND Microbiome jointly complete: {len(both_complete)}/{len(dynamic_ids)}")

Gut AND Microbiome jointly complete: 42/45


In [33]:
# --- Static-table schema cross-check against each table's own dictionary ---

# Bio has three columns literally named "Time (t)"; pandas auto-suffixes the repeats
# (".1", ".2") on read, so that suffix is stripped before dictionary matching.
def build_dictionary_lookup(dictionary_df):
    lookup = {}
    for _, row in dictionary_df.iterrows():
        lookup.setdefault(row["normalized_field"], row["Field"])
    return lookup

# Build a normalized-name lookup for each static table's dictionary.
bio_lookup = build_dictionary_lookup(bio_dictionary_df)
gut_lookup = build_dictionary_lookup(gut_dictionary_df)
microbes_lookup = build_dictionary_lookup(microbes_dictionary_df)

# Cross-check each static table's actual columns against its dictionary lookup.
static_schema_rows = []
for name, df, lookup in [
    ("bio.csv", static_tables["bio.csv"], bio_lookup),
    ("gut_health_test.csv", static_tables["gut_health_test.csv"], gut_lookup),
    ("microbes.csv", static_tables["microbes.csv"], microbes_lookup),
]:
    for col in df.columns:
        if col == "subject":
            continue
        base_col = re.sub(r"\.\d+$", "", col)
        norm = normalize_column_name(base_col)
        static_schema_rows.append({
            "table": name,
            "raw_column_name": col,
            "dictionary_field_match": lookup.get(norm),
        })

static_schema_map = pd.DataFrame(static_schema_rows)
static_schema_map["matched"] = static_schema_map["dictionary_field_match"].notna()

# Report match counts per table and list any unmatched columns.
print(static_schema_map.groupby("table")["matched"].agg(["sum", "count"]).to_string())
unmatched = static_schema_map[~static_schema_map["matched"]]
if not unmatched.empty:
    print(unmatched[["table", "raw_column_name"]].to_string(index=False))

                      sum  count
table                           
bio.csv                23     23
gut_health_test.csv    22     22
microbes.csv         1979   1979


In [34]:
# --- Documented static-data conflicts (not resolved here; see the data interpretation notes below) ---

bio_df = static_tables["bio.csv"]

hba1c_range = (bio_df["A1c PDL (Lab)"].min(), bio_df["A1c PDL (Lab)"].max())
n_ldl_error = int((bio_df["LDL (Cal)"] == 800).sum())
n_vldl_error = int((bio_df["VLDL (Cal)"] == 400).sum())
n_cho_hdl_at_max = int((bio_df["Cho/HDL Ratio"] == 400).sum())
duplicate_time_columns = [c for c in bio_df.columns if re.match(r"^Time \(t\)(\.\d+)?$", c)]

print(f"A1c PDL (Lab) observed range: {hba1c_range} (dictionary declares 4.6-8.5 but describes the unit as mmol/mol)")
print(pd.Series({"LDL at dictionary error value": n_ldl_error, "VLDL at dictionary error value": n_vldl_error, "Cho/HDL at range maximum": n_cho_hdl_at_max, "Bio Time columns": len(duplicate_time_columns)}).to_string())

A1c PDL (Lab) observed range: (np.float64(4.6), np.float64(8.5)) (dictionary declares 4.6-8.5 but describes the unit as mmol/mol)
LDL at dictionary error value     1
VLDL at dictionary error value    1
Cho/HDL at range maximum          1
Bio Time columns                  3


In [35]:
# --- HbA1c-based metabolic-status grouping under a percent interpretation ---

def hba1c_percent_group(value):
    if value < 5.7:
        return "Healthy"
    if value < 6.5:
        return "Pre-diabetes"
    return "Type 2 Diabetes"

hba1c_groups = bio_df["A1c PDL (Lab)"].map(hba1c_percent_group)
print(hba1c_groups.value_counts().to_string())

A1c PDL (Lab)
Pre-diabetes       16
Healthy            15
Type 2 Diabetes    14


In [36]:
# --- Raw bio.csv summary statistics (all 45 rows, no exclusions, no unit conversion) ---

bio_stats = {
    "A1c PDL (Lab) mean": bio_df["A1c PDL (Lab)"].mean(),
    "Insulin mean": bio_df["Insulin "].mean(),
    "BMI sample SD (ddof=1)": bio_df["BMI"].std(ddof=1),
    "BMI population SD (ddof=0)": bio_df["BMI"].std(ddof=0),
    "Age mean": bio_df["Age"].mean(),
    "Fasting GLU mean": bio_df["Fasting GLU - PDL (Lab)"].mean(),
}
for label, value in bio_stats.items():
    print(f"{label}: {value}")

A1c PDL (Lab) mean: 6.122222222222222
Insulin mean: 14.16666666666667
BMI sample SD (ddof=1): 6.728244449116464
BMI population SD (ddof=0): 6.6530661739129755
Age mean: 48.111111111111114
Fasting GLU mean: 120.68888888888888


In [37]:
# --- Build subject_master: cohort membership, integrity flags and key Bio identity fields ---

# This is a subject roster; static feature sets are joined to the dynamic data in data alignment.

# Add per-table "fully empty" integrity flags.
subject_master = cohort_membership.copy()
subject_master["gut_fully_empty"] = subject_master.index.isin(gut_empty_set)
subject_master["microbes_fully_empty"] = subject_master.index.isin(microbes_empty_set)

# Join in a few key Bio identity fields.
bio_key_fields = bio_df.set_index("subject")[["Age", "Gender", "BMI", "A1c PDL (Lab)"]]
subject_master = subject_master.join(bio_key_fields)

### Data interpretation notes

- **HbA1c:** retained without unit conversion (treated as percent, not the dictionary's mmol/mol).
- **Cho/HDL Ratio:** the value 400 is retained and flagged as a candidate data-quality issue, not treated as an error.
- **Repeated `Time (t)` columns:** the three `bio.csv` fingerstick-time columns are renamed by position during cleaning.
- **Gut-health-only subjects:** subjects 24 and 25 have no dynamic CGM records and are excluded from analyses requiring dynamic data.
- **Empty static rows:** fully empty rows are handled during static-data integration (the data-alignment notebook), not here.
- **Microbiome variables:** binary detection indicators, not abundance; missing values are not interpreted as absence.

## Development and test scope

Define splits from subject IDs and timestamp spans, without glucose values or model results. Protocol A uses each subject's elapsed-time 60/20/20 split; Protocol B assigns whole subjects to 27/9/9 train/validation/test groups (seed 42).

`split_scope_v1.csv` also defines the shared exploratory scope: Protocol B training subjects during their Protocol A training period.


In [39]:
# --- Protocol A: Within-subject Future Prediction (per-person temporal split) ---

# Split by elapsed *time*, not by row index, so occasional gaps (subjects 7, 18) do not
# silently shift the intended time boundary.
WITHIN_SUBJECT_TRAIN_FRACTION = 0.60
WITHIN_SUBJECT_VAL_FRACTION = 0.20  # remaining 0.20 is test

protocol_a_rows = []
for pid, df in participant_dfs.items():
    timestamps = parsed_timestamps[pid].dropna().sort_values()
    start_time, end_time = timestamps.iloc[0], timestamps.iloc[-1]
    span = end_time - start_time

    train_end_time = start_time + span * WITHIN_SUBJECT_TRAIN_FRACTION
    val_end_time = start_time + span * (WITHIN_SUBJECT_TRAIN_FRACTION + WITHIN_SUBJECT_VAL_FRACTION)

    n_train = int((timestamps <= train_end_time).sum())
    n_val = int(((timestamps > train_end_time) & (timestamps <= val_end_time)).sum())
    n_test = int((timestamps > val_end_time).sum())

    protocol_a_rows.append({
        "subject_id": pid,
        "start_time": start_time,
        "end_time": end_time,
        "protocol_a_train_end_time": train_end_time,
        "protocol_a_val_end_time": val_end_time,
        "protocol_a_n_train_rows": n_train,
        "protocol_a_n_val_rows": n_val,
        "protocol_a_n_test_rows": n_test,
    })

protocol_a_scope = pd.DataFrame(protocol_a_rows).set_index("subject_id")
print(protocol_a_scope[["protocol_a_n_train_rows", "protocol_a_n_val_rows", "protocol_a_n_test_rows"]].sum().to_string())

protocol_a_n_train_rows    407439
protocol_a_n_val_rows      140063
protocol_a_n_test_rows     140078


In [40]:
# --- Protocol B: Unseen-subject Generalization (subject-level split) ---

UNSEEN_SUBJECT_SPLIT_SEED = 42
UNSEEN_SUBJECT_N_TRAIN = 27
UNSEEN_SUBJECT_N_VAL = 9
UNSEEN_SUBJECT_N_TEST = 9

all_subjects_sorted = sorted(dynamic_ids)
assert len(all_subjects_sorted) == UNSEEN_SUBJECT_N_TRAIN + UNSEEN_SUBJECT_N_VAL + UNSEEN_SUBJECT_N_TEST


def assign_unseen_subject_groups(subjects, seed):
    """Shuffle `subjects` with a fixed seed and assign whole subjects to train/val/test."""
    shuffled = np.random.default_rng(seed).permutation(subjects)
    groups = {}
    for pid in shuffled[:UNSEEN_SUBJECT_N_TRAIN]:
        groups[int(pid)] = "train"
    for pid in shuffled[UNSEEN_SUBJECT_N_TRAIN:UNSEEN_SUBJECT_N_TRAIN + UNSEEN_SUBJECT_N_VAL]:
        groups[int(pid)] = "val"
    for pid in shuffled[UNSEEN_SUBJECT_N_TRAIN + UNSEEN_SUBJECT_N_VAL:]:
        groups[int(pid)] = "test"
    scope = pd.Series(groups, name="protocol_b_group").sort_index()
    scope.index.name = "subject_id"
    return scope

protocol_b_scope = assign_unseen_subject_groups(all_subjects_sorted, UNSEEN_SUBJECT_SPLIT_SEED)

In [41]:
# --- Combine into split_scope_v1, with the cross-protocol safe-exploration range ---

# Only subjects assigned to Protocol B's train group contribute a "common training range";
# a Protocol B val/test subject's Protocol A training portion must NOT be used for any
# cross-protocol exploration, even though it technically exists.

def build_split_scope(protocol_b_groups):
    """Join the Protocol A boundaries with a Protocol B assignment and mark the common training range."""
    scope = protocol_a_scope.join(protocol_b_groups)
    scope["common_training_range_available"] = scope["protocol_b_group"] == "train"
    scope["common_training_range_end_time"] = scope["protocol_a_train_end_time"].where(
        scope["common_training_range_available"]
    )
    return scope

split_scope_v1 = build_split_scope(protocol_b_scope)
print(f"Subjects with a common safe training range for cross-protocol EDA: "
      f"{int(split_scope_v1['common_training_range_available'].sum())}")

Subjects with a common safe training range for cross-protocol EDA: 27


In [42]:
# --- Integrity checks on split_scope_v1 ---

protocol_a_row_total = (
    split_scope_v1["protocol_a_n_train_rows"]
    + split_scope_v1["protocol_a_n_val_rows"]
    + split_scope_v1["protocol_a_n_test_rows"]
)
assert protocol_a_row_total.eq(coverage_summary["n_rows"]).all(), "Protocol A segments must partition each subject's rows."
assert split_scope_v1["protocol_b_group"].notna().all() and len(split_scope_v1) == len(dynamic_ids),\
    "Protocol B must assign every subject exactly once."
assert split_scope_v1["protocol_b_group"].value_counts().to_dict() == {
    "train": UNSEEN_SUBJECT_N_TRAIN, "val": UNSEEN_SUBJECT_N_VAL, "test": UNSEEN_SUBJECT_N_TEST,
}, "Protocol B group sizes must be 27/9/9."

# --- Persist split_scope_v1 (seed 42, the primary split) ---

split_scope_v1_path = DATA_QUALITY_DIR / "split_scope_v1.csv"
split_scope_v1.to_csv(split_scope_v1_path)